# Analysis SITCOM-1948: ComCam

Analysis from SITCOM-1884: Rotator Torque Analysis with Pancake Wrap did not show big differences in the rotator torques before/after the installation of the pancake wrap on the top-end integration assembly.

We want to dive deeper and perform an analysis “per movement”. Similar to SITCOM-1946: CamHex Torque "per movement" Analysis with Pancake Wrap, we want to convert the torque profile into metrics that we can use for statistical analysis.

The idea is to use `slew_events` as the movement and analyse the data for only those events

This notebook will focus on the ComCam data, taken during the ComCam campaign. For the analysis on the LSSTCam rotator and CCW data, refer to the second notebook from this ticket. 

In [ ]:
#%matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np
from scipy.integrate import simpson

from astropy.time import Time
from pathlib import Path
from datetime import datetime

from lsst.summit.utils.tmaUtils import (
    TMAEventMaker,
    TMAState,
)
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

Here we will do the analysis for the slew events

In [ ]:
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

# Set global font size for labels, titles, and ticks
plt.rcParams.update({
    "font.size": 14,  
    "axes.labelsize": 16,  
    "axes.titlesize": 18,  
    "xtick.labelsize": 14,  
    "ytick.labelsize": 14,  
})

event_maker = TMAEventMaker()
efd_client = makeEfdClient()

# Data Analysis

## Single Day

We are looking for data during one day.  
This comes with the first filter below.  
There are situations where `actualTorquePercentage` reports only `None` values. We need to filter these out. 

In [ ]:
day_obs = 20241207
all_events = event_maker.getEvents(day_obs)
slew_events = [e for e in all_events if e.type == TMAState.SLEWING]

In [ ]:
evt = all_events[0]
print(evt)

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0","torque1"],
    event=evt,
)

In [ ]:
ccw = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTMount.cameraCableWrap",
    columns=["actualTorquePercentage0", "actualTorquePercentage1", "actualPosition", "actualPositionTimestamp"],
    event=evt,
)

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTPtg.mountPosition",
    columns=["rotatorActualPosition"],
    event=evt,
)

In [ ]:
print(slew_events[0])  # Checking structure of the first event


Here we will make the dataframe for the movements

In [ ]:
movements = []

for i, evt in enumerate(slew_events):
    # Fetch data for each slew event
    df = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.cameraCableWrap",
        columns=["actualPosition", "actualTorquePercentage0", "actualTorquePercentage1"],
        event=evt,
        warn=False,
    )

    if df.empty:
        print(f"Skipping empty dataframe for event {i + 1}")
        continue

    # Extract timestamps
    move_start = evt.begin.utc.value  # Convert Time object to Unix timestamp
    move_end = evt.end.utc.value      # Convert Time object to Unix timestamp
    time_series = pd.date_range(start=pd.to_datetime(move_start, unit='s'),
                                end=pd.to_datetime(move_end, unit='s'),
                                periods=len(df))  # Generate evenly spaced timestamps

    movements.append({
        "movement_id": i + 1,
        "move_start": move_start,
        "move_end": move_end,
        "time": time_series,  # Add generated timestamps
        "torque0": df["actualTorquePercentage0"].values if "actualTorquePercentage0" in df else None,
        "torque1": df["actualTorquePercentage1"].values if "actualTorquePercentage1" in df else None,
        "actualPosition": df["actualPosition"].values if "actualPosition" in df else None,
    })

movement_df = pd.DataFrame(movements)

print(movement_df.head())


In [ ]:
movement_df

# Plotting the first 10 movements 

Here we'll plot the first 10 movements to see how they look 

In [ ]:
num_movements = min(10, len(movement_df))  

for i in range(num_movements):
    movement = movement_df.iloc[i] 
    
    fig, ax1 = plt.subplots(figsize=(8, 5))  
    ax1.plot(movement["time"], movement["torque0"], 'C0-', label="Torque 0")
    ax1.plot(movement["time"], movement["torque1"], 'C1-', label="Torque 1")
    ax1.set_xlabel("Time (UTC)")
    ax1.set_ylabel("Torque [%]")
    ax1.tick_params(axis='y', labelcolor='black')

    ax2 = ax1.twinx()  
    ax2.plot(movement["time"], movement["actualPosition"], 'g--', label="Rotator Position")
    ax2.set_ylabel("Rotator Position", color='g')
    ax2.tick_params(axis='y', labelcolor='g')

    ax1.set_title(f"Slew {movement['movement_id']}")
    fig.autofmt_xdate()  
    
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")

    plt.grid(":", alpha=0.25)
    plt.show()



Let's filter the data only when the rotator is changing the position

In [ ]:
movements = []

for i, evt in enumerate(slew_events):

    df = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.cameraCableWrap",
        columns=["actualPosition", "actualTorquePercentage0", "actualTorquePercentage1"],
        event=evt,
        warn=False,
    )

    if df.empty:
        print(f"Skipping empty dataframe for event {i + 1}")
        continue

    position_change = df["actualPosition"].max() - df["actualPosition"].min()
    
    if position_change <= 1:
        continue

    move_start = evt.begin.utc.value  
    move_end = evt.end.utc.value      
    time_series = pd.date_range(start=pd.to_datetime(move_start, unit='s'),
                                end=pd.to_datetime(move_end, unit='s'),
                                periods=len(df))  

    movements.append({
        "movement_id": i + 1,
        "move_start": move_start,
        "move_end": move_end,
        "time": time_series,  
        "torque0": df["actualTorquePercentage0"].values if "actualTorquePercentage0" in df else None,
        "torque1": df["actualTorquePercentage1"].values if "actualTorquePercentage1" in df else None,
        "actualPosition": df["actualPosition"].values if "actualPosition" in df else None,
        "position_change": position_change,  
    })

movement_df = pd.DataFrame(movements)

print(movement_df.head())


In [ ]:
movement_df

# Plotting first 10 slews

Now let's plot the first 10 slew events

In [ ]:
num_movements = min(10, len(movement_df))  

for i in range(num_movements):
    movement = movement_df.iloc[i]  
    
    fig, ax1 = plt.subplots(figsize=(8, 5))

    line1, = ax1.plot(movement["time"], movement["torque0"], color='C0', label="Torque 0")
    line2, = ax1.plot(movement["time"], movement["torque1"], color='C1', label="Torque 1")
    
    ax1.set_xlabel("Time (UTC)")
    ax1.set_ylabel("Torque (%)")
    ax1.tick_params(axis='y')
    plt.xticks(rotation=45)

    ax2 = ax1.twinx()
    line3, = ax2.plot(movement["time"], movement["actualPosition"], color='C2', linestyle="--", label="Rotator Position")
    
    ax2.set_ylabel("CCW Position (deg)", labelpad=15)
    ax2.tick_params(axis='y')

    plt.title(f"ComCam: Torques and CCW position for the slew {movement['movement_id']})")
    
    ax1.legend(handles=[line1, line2], labels=["Torque 0", "Torque 1"], loc="upper left")
    ax2.legend(handles=[line3], labels=["CCW Position"], loc="upper right")
    
    ax1.grid(":", alpha=0.25)
    plt.tight_layout()

    plt.show()


# Plotting unbiased slews

Let's remove the bias, calculate the area under the torque curve, and add it to data frame

In [ ]:
if len(movement_df) > 0:
    areas = []  

    for i, row in movement_df.iterrows():
        time = row["time"]
        torque0 = row["torque0"]
        torque1 = row["torque1"]
        position = row["actualPosition"]

        torque0 -= np.mean(torque0)
        torque1 -= np.mean(torque1)

        area_torque0 = simpson(np.abs(torque0), dx=1)  
        area_torque1 = simpson(np.abs(torque1), dx=1)  

        total_area = area_torque0 + area_torque1  
        areas.append(total_area)  

        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(time, torque0, color="C0", label="Torque 0 (Bias Removed)")
        ax.plot(time, torque1, color="C1", label="Torque 1 (Bias Removed)")

        ax.set_xlabel("Time [UTC]")
        ax.set_ylabel("Torque (%)")
        plt.xticks(rotation=45)
        ax.set_title(f"ComCam: Torque vs Time for Movement {i + 1}")

        ax2 = ax.twinx()
        ax2.plot(time, position, color="C2", label="Position", linestyle="--")
        ax2.set_ylabel("CCW Position (deg)")

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.xaxis.set_major_locator(mdates.MinuteLocator())

        handles, labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(handles + handles2, labels + labels2, loc="upper left")

        ax.grid(which="major", axis="both", linestyle="--", alpha=0.25)
        
        area_text = f"Total Area: {total_area:.2f} %·s"
        ax.text(0.95, 0.05, area_text, transform=ax.transAxes,
                ha="right", va="bottom", fontsize=12, color="black",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", boxstyle="round,pad=0.5"))

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    movement_df["area (% s)"] = areas

else:
    print("No valid movement pairs found.")


Let's calculate the movement size according to the ccw position 

In [ ]:
movement_sizes = []

for i, row in movement_df.iterrows():
    position = row['actualPosition']
    
    movement_size = np.abs(position.max() - position.min())
    movement_sizes.append(movement_size)

movement_df['movement_size (degrees)'] = movement_sizes

print(movement_df[['movement_size (degrees)']])


# Plotting the area VS movement size

In [ ]:
if 'area (% s)' in movement_df.columns and 'movement_size (degrees)' in movement_df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist2d(movement_df['movement_size (degrees)'], movement_df['area (% s)'], bins=30, cmap='Blues')
    
    ax.set_xlabel('Movement Size (degrees)')
    ax.set_ylabel('Area (%·s)')
    ax.set_title('ComCam: Histogram of Area vs Movement Size for the ccw')
    
    cbar = plt.colorbar(ax.collections[0], ax=ax)
    cbar.set_label('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not found in the DataFrame.")


# Fitting the area versus movement size

In [ ]:
if 'area (% s)' in movement_df.columns and 'movement_size (degrees)' in movement_df.columns:
    x = movement_df['movement_size (degrees)']
    y = movement_df['area (% s)']
    
    # Perform linear regression (best-fit line)
    coefficients = np.polyfit(x, y, 1)  
    poly = np.poly1d(coefficients)

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, color='C0', alpha=0.6, label='Data points')
    
    plt.plot(x, poly(x), color='red', linestyle='--', label='Best fit line')

    plt.xlabel('Movement Size (degrees)')
    plt.ylabel('Area (%·s)')
    plt.title('ComCam: Area vs Movement Size for the CCW')

    slope = coefficients[0]
    intercept = coefficients[1]
    equation = f'Fit: y = {slope:.6f}x + {intercept:.6f}'

    plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', color='black')

    plt.grid(":", alpha=0.25)

    plt.legend()

    plt.tight_layout()
    plt.show()

else:
    print("Required columns not found in the DataFrame.")


Polinom fit

In [ ]:
if 'area (% s)' in movement_df.columns and 'movement_size (degrees)' in movement_df.columns:
    x = movement_df['movement_size (degrees)']
    y = movement_df['area (% s)']

    coefficients = np.polyfit(x, y, 3)  
    poly = np.poly1d(coefficients)

    x_fit = np.linspace(x.min(), x.max(), 500)  
    y_fit = poly(x_fit)

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, color='C0', alpha=0.6, label='Data points')

    plt.plot(x_fit, y_fit, color='red', linestyle='--', label='Polynomial fit (degree 3)')

    plt.xlabel('Movement Size (degrees)')
    plt.ylabel('Area (%·s)')
    plt.title('ComCam: Area vs Movement Size with Polynomial Fit (Degree 3) - CCW')

    equation = f'Fit: y = {coefficients[0]:.2e}x³ + {coefficients[1]:.2e}x² + {coefficients[2]:.2e}x + {coefficients[3]:.2e}'

    plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', color='black')

    plt.grid(":", alpha=0.25)

    plt.legend()

    plt.tight_layout()
    plt.show()

else:
    print("Required columns not found in the DataFrame.")


# Let's try some metrics, like Effitiency ratio: 

This metric tells you how much torque effort (area under the curve) is needed per degree of rotator movement.

Eq: ER=area/movenment size

In [ ]:
movement_df["efficiency_ratio"] = movement_df["area (% s)"] / movement_df["movement_size (degrees)"]

print(movement_df["efficiency_ratio"].describe())

Let's do the histogram. If the values cluster around a specific range, it means most movements require similar torque per degree.

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(movement_df["efficiency_ratio"], bins=15, color="C0", alpha=0.7, edgecolor="black")
plt.xlabel("Efficiency Ratio (%·s per degree)")
plt.ylabel("Frequency")
plt.title("ComCam: Distribution of Efficiency Ratios for the rotator")
plt.grid(axis="y", linestyle="--", alpha=0.25)
plt.show()

ER VS Movement size. If we see a downward trend, it means larger movements are more efficient (less torque per degree).

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(movement_df["movement_size (degrees)"], movement_df["efficiency_ratio"], color="C0", alpha=0.7)
plt.xlabel("Movement Size (degrees)")
plt.ylabel("Efficiency Ratio (%·s per degree)")
plt.title("ComCam: Efficiency Ratio vs Movement Size for the CCW")
plt.grid(True, linestyle="--", alpha=0.25)
plt.show()